## Parse Structured XML Generated by Grobid

In [121]:
import json
import logging
import Levenshtein
import os
import requests

from bs4 import BeautifulSoup

In [122]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

In [123]:
TEI_XML_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/tei/'
TEI_XML_CROSSREF_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/tei_crossref_citations/'
GROUPED_REFS_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/grouped_refs_full/'
REFS_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/refs_selected/'

LEVENSHTEIN_TITLE_DISTANCE_THRESHOLD = 5

In [124]:
def extract_grouped_refs(soup):
    """
    Extract references grouped by occurrence in the same paragraph of the paper.
    """
    text = soup.find('text')
    grouped_refs = []
    for child_tag in text.body.children:
        # Skip empty lines and non-text tags (e.g., notes)
        if child_tag.name != 'div':
            continue

        # Extract title if present
        title = child_tag.head.text if child_tag.head else ''

        # Group all references from a single div
        ref_tags = child_tag.find_all('ref', attrs={"type": "bibr"})
        refs = [(ref_tag.get('target', None), ref_tag.text)
                for ref_tag in ref_tags]
        grouped_refs.append({'title': title, 'references': refs})
    return grouped_refs

In [125]:
def extract_refs(soup):
    """
    Extract list of references for the paper.
    """
    bibliography = soup.find('listBibl')
    refs = {}
    for bibl_struct in bibliography.children:
        # Skip empty lines
        if bibl_struct.name != 'biblStruct':
            continue

        ref_id = bibl_struct['xml:id']
        ref_text = ''
        if bibl_struct.analytic and bibl_struct.analytic.title:
            ref_text = bibl_struct.analytic.title.text
        refs[ref_id] = ref_text
    return refs

In [126]:
def parse_xml(filename):
    with open(filename, 'r') as xml_file:
        soup = BeautifulSoup(xml_file, 'xml')
    
    grouped_refs = extract_grouped_refs(soup)
    refs = extract_refs(soup)
    return grouped_refs, refs

In [127]:
def export_to_json(data, filename):
    with open(filename, 'w') as f:
        json.dump(data, f, indent=4)

In [134]:
def process_xml(filename):
    pmid = filename.split('.')[0]
    xml_raw = os.path.join(TEI_XML_FOLDER, filename)
    xml_crossref = os.path.join(TEI_XML_CROSSREF_FOLDER, filename)

    grouped_refs_raw, refs_raw = parse_xml(xml_raw)
    grouped_refs_crossref, refs_crossref = parse_xml(xml_crossref)
    
    if grouped_refs_raw != grouped_refs_crossref:
        print('Grouped refs raw vs crossref differ')
        
    refs_selected = {}
    for ref_id, ref_raw in refs_raw.items():
        ref_fixed = refs_crossref[ref_id]
        distance = Levenshtein.distance(ref_raw.lower(), ref_fixed.lower())
        
        selected = False
        reason = ''
        if distance > LEVENSHTEIN_TITLE_DISTANCE_THRESHOLD and ref_raw:
            print(ref_raw)
            print(ref_fixed)
            match = input('Is this the key paper?')
            if match == '':
                selected = True
                reason = ref_raw
                selected_ref = {
                    'title': ref_fixed,
                    'selected': selected,
                    'reason': reason
                }

                refs_selected[ref_id] = selected_ref
    
#     grouped_refs_filename = os.path.join(GROUPED_REFS_FOLDER, f'{pmid}.json')
#     export_to_json(grouped_refs_raw, grouped_refs_filename)
        
    refs_filename = os.path.join(REFS_FOLDER, f'{pmid}.json')
    export_to_json(refs_selected, refs_filename)

In [144]:
for folder in [GROUPED_REFS_FOLDER, REFS_FOLDER]:
    if not os.path.exists(folder):
        os.mkdir(folder)
for filename in os.listdir(TEI_XML_FOLDER):
    if filename.endswith('.xml'):
        logging.info(f'Processing {filename}')
        process_xml(filename)

2021-02-07 01:27:15,147 INFO: Processing 26675821.tei.xml


Die Gastraea-Theorie, die phylogenetische Classification des Thierreiches und die Homologie der Keimblätter
Zwanzigster Vortrag. Phylogenetische Classification des Thierreichs. Gastraea-Theorie.
Is this the key paper?N
Nerve nets and conducting systems in sea anemones: two pathways excite tentacle contractions in Calliactis parasitica
Nerve nets and conducting systems in sea anemones: Coordination of ipsilateral and contralateral contractions inProtanthea simplex
Is this the key paper?N
Control of mouth opening and pharynx protrusion during feeding in the sea anemone Calliactis parasitica
Interactions Between Conducting Systems in the Sea Anemone Calliactis Parasitica
Is this the key paper?N
The dynamic genome of Hydra
Human genome at ten: The sequence explosion
Is this the key paper?N
Cellular resolution expression profiling reveals common origin of annelid mushroom bodies and vertebrate pallium
Profiling by Image Registration Reveals Common Origin of Annelid Mushroom Bodies and Verte

2021-02-07 01:27:36,961 INFO: Processing 30459365.tei.xml


This is the largest-scale study of circadian rhythms of gene expression in human post-mortem brain tissue, revealing age-related decline in the expression of core circadian genes and the emergence of other rhythmic pathways in older subjects
Effects of aging on circadian patterns of gene expression in the human prefrontal cortex
Is this the key paper?
Shift work disorder: overview and diagnosis
Shift Work Disorder
Is this the key paper?N
Reproductive health outcomes among female flight attendants: an exploratory study
Reproductive Health Outcomes Among Female Flight Attendants
Is this the key paper?N
Sleep in infants and young children: part two: common sleep problems
Sleep in infants and young children
Is this the key paper?N
This randomized intervention study reports that preterm infants (<31 weeks) receiving cycled light (11 hours on and 11 hours off, with 1 transition hour for shift changes) during hospital care
Preterm infants born at less than 31 weeks’ gestation have improved gr

2021-02-07 01:28:03,456 INFO: Processing 26656254.tei.xml


...

Concerning the origin of malignant tumours by Theodor Boveri
Concerning the Origin of Malignant Tumours by Theodor Boveri. Translated and annotated by Henry Harris
Is this the key paper?N
A mammalian endonuclease specific for apurinic sites in doublestranded deoxyribonucleic acid. I. Purification and general properties
A Mammalian Endonuclease Specific for Apurinic Sites in Double-stranded Deoxyribonucleic Acid
Is this the key paper?N
Cellular hypersensitivity to neocarzinostatin in ataxiatelangiectasia skin fibroblasts
The response of ataxia-telangiectasia homozygous and heterozygous skin fibroblasts to neocarzinostatin
Is this the key paper?N
The multistep nature of cancer development
Breast Cancer Multistep Development
Is this the key paper?N
Errors in DNA replication as a basis of malignant changes
On Mutagenic DNA Polymerases and Malignancy
Is this the key paper?N
Mutator phenotype may be required for multistage carcinogenesis
Mutator Phenotype
Is this the key paper?N
Cancer Genom

2021-02-07 01:34:33,302 INFO: Processing 31836872.tei.xml


This study reveals that memory B cells and longlived plasma cells are produced in different phases of the GC reaction, with memory B cell formation preceding the formation of plasma cells
A Temporal Switch in the Germinal Center Determines Differential Output of Memory B and Plasma Cells
Is this the key paper?
This study shows that owing to their high affinity and ability to be activated in the presence of neutralizing antibodies, isotype-switched IgG-expressing memory B cells prevail during the early phase of immune memory; however, once they disappear due to a shorter life span
Different B Cell Populations Mediate Early and Late Memory During an Endogenous Immune Response
Is this the key paper?


2021-02-07 01:34:35,202 INFO: Processing 30644449.tei.xml


The inflammasome: a molecular platform triggering activation of inflammatory caspases and processing of proIL-β
The Inflammasome
Is this the key paper?N
This report shows that DNA-activated STINGdependent cell death occurs in human myeloid cells through lysosomal cell death and upstream of activation of the NLRP3 inflammasome
The DNA Inflammasome in Human Myeloid Cells Is Initiated by a STING-Cell Death Program Upstream of NLRP3
Is this the key paper?
Type I interferon-mediated autoinflammation due to DNase II deficiency
Type I interferon–mediated monogenic autoinflammation: The type I interferonopathies, a conceptual overview
Is this the key paper?N
This is the first demonstration of induction of necroptosis in a STING-dependent manner
Induction of necroptotic cell death by viral activation of the RIG-I or STING pathway
Is this the key paper?
The authors of this study identify a virus-induced STING-dependent apoptosis pathway dependent on IRF3-BAX signalling and show that this pathway

## Validating Grouped References

In [73]:
GROUPED_REFS_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/grouped_refs_validated/'
REFS_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/refs_validated/'

In [69]:
def flatten_grouped_refs(grouped_refs):
    refs = []
    for el in grouped_refs:
        if isinstance(el, dict):
            refs.extend(flatten_grouped_refs(el['references']))
        else:
            assert len(el) == 2
            # Clean numeric ids, e.g. 'REF. 37' -> 37
            grobid_id, numeric_id = el
            numeric_id = int(''.join(list(filter(lambda c: c.isdigit(), numeric_id))))
            # Check Grobid ID: should be '#bXXX'
            if grobid_id:
                assert grobid_id[:2] == '#b'
                grobid_number = int(grobid_id[2:])  # should be a valid int
            refs.append([grobid_id, numeric_id])
    return refs

### 1. JSON is valid - OK

In [75]:
import json
import os

for file in sorted(os.listdir(GROUPED_REFS_FOLDER)):
    print(file, end='\t')
    filename = os.path.join(GROUPED_REFS_FOLDER, file)
    with open(filename, 'r') as f:
        data = json.load(f)
    print('OK')

26580716.json	OK
26580717.json	OK
26656254.json	OK
26667849.json	OK
26675821.json	OK
26678314.json	OK
26688349.json	OK
26688350.json	OK
27677859.json	OK
27677860.json	OK
27834397.json	OK
27834398.json	OK
27890914.json	OK
27904142.json	OK
27916977.json	OK
28003656.json	OK
28792006.json	OK
28852220.json	OK
28853444.json	OK
28920587.json	OK
29147025.json	OK
29170536.json	OK
29213134.json	OK
29321682.json	OK
30108335.json	OK
30390028.json	OK
30459365.json	OK
30467385.json	OK
30578414.json	OK
30644449.json	OK
30679807.json	OK
30842595.json	OK
31686003.json	OK
31806885.json	OK
31836872.json	OK
31937935.json	OK
32005979.json	OK
32020081.json	OK
32042144.json	OK
32699292.json	OK


In [74]:
for file in sorted(os.listdir(REFS_FOLDER)):
    print(file, end='\t')
    filename = os.path.join(REFS_FOLDER, file)
    with open(filename, 'r') as f:
        data = json.load(f)
    print('OK')

26580716.json	OK
26580717.json	OK
26656254.json	OK
26667849.json	OK
26675821.json	OK
26678314.json	OK
26688349.json	OK
26688350.json	OK
27677859.json	OK
27677860.json	OK
27834397.json	OK
27834398.json	OK
27890914.json	OK
27904142.json	OK
27916977.json	OK
28003656.json	OK
28792006.json	OK
28852220.json	OK
28853444.json	OK
28920587.json	OK
29147025.json	OK
29170536.json	OK
29213134.json	OK
29321682.json	OK
30108335.json	OK
30390028.json	OK
30459365.json	OK
30467385.json	OK
30578414.json	OK
30644449.json	OK
30679807.json	OK
30842595.json	OK
31686003.json	OK
31806885.json	OK
31836872.json	OK
31937935.json	OK
32005979.json	OK
32020081.json	OK
32042144.json	OK
32699292.json	OK


### 2. Looking for Errors in Grouped References JSONs

* null / None values of Grobid ID (`#bXXX`) occur if reference was not found by Grobid
* Grobid ID and numeric reference ID should correspond one to another

In [71]:
import json
import os

print('\t\tNO_NULL_IDS\tUNIQUE_IDS')
for file in sorted(os.listdir(GROUPED_REFS_FOLDER)):
    print(file, end='\t')
    filename = os.path.join(GROUPED_REFS_FOLDER, file)
    with open(filename, 'r') as f:
        grouped_refs = json.load(f)
    flat_grouped_refs = flatten_grouped_refs(grouped_refs)
    
    # Check that no 'null' refs are present in grouped reference files (might occur due to Grobid errors)
    null_refs = False
    for ref in flat_grouped_refs:
        # Check if flatten function works
        assert type(ref) == list
        assert len(ref) == 2
        grobid_id, numeric_id = ref
        if not grobid_id:
            null_refs = True
            break
    if null_refs:
        print('ERROR', end='\t\t')
    else:
        print('OK', end='\t\t')
        
    # Check that grobid_id <-> numeric_id is a bijection
    grobid2numeric = {}
    numeric2grobid = {}
    for ref in flat_grouped_refs:
        grobid_id, numeric_id = ref
        
        if grobid_id not in grobid2numeric:
            grobid2numeric[grobid_id] = set()
        grobid2numeric[grobid_id].add(numeric_id)
        
        if numeric_id not in numeric2grobid:
            numeric2grobid[numeric_id] = set()
        numeric2grobid[numeric_id].add(grobid_id)
    
    unique_ids = True
    error_ids = []
    for gid, nids in grobid2numeric.items():
        if gid == 'null':
            continue
        if len(nids) > 1:
            error_ids.append((gid, nids))
    for nid, gids in numeric2grobid.items():
        if len(gids) > 1:
            error_ids.append((nid, gids))
    if unique_ids:
        print('OK')
    else:
        print(f"ERROR {', '.join([error_ids])}")

		NO_NULL_IDS	UNIQUE_IDS
26580716.json	OK		OK
26580717.json	OK		OK
26656254.json	OK		OK
26667849.json	OK		OK
26675821.json	OK		OK
26678314.json	OK		OK
26688349.json	OK		OK
26688350.json	OK		OK
27677859.json	OK		OK
27677860.json	OK		OK
27834397.json	OK		OK
27834398.json	ERROR		OK
27890914.json	OK		OK
27904142.json	OK		OK
27916977.json	OK		OK
28003656.json	OK		OK
28792006.json	OK		OK
28852220.json	OK		OK
28853444.json	OK		OK
28920587.json	OK		OK
29147025.json	OK		OK
29170536.json	OK		OK
29213134.json	OK		OK
29321682.json	OK		OK
30108335.json	OK		OK
30390028.json	OK		OK
30459365.json	OK		OK
30467385.json	OK		OK
30578414.json	OK		OK
30644449.json	OK		OK
30679807.json	OK		OK
30842595.json	OK		OK
31686003.json	OK		OK
31806885.json	OK		OK
31836872.json	ERROR		OK
31937935.json	OK		OK
32005979.json	OK		OK
32020081.json	OK		OK
32042144.json	ERROR		OK
32699292.json	OK		OK


## Obtaining the Clustering with PMIDs

In [78]:
import gzip
import json
import logging
import Levenshtein
import os

from pysrc.papers.analyzer import KeyPaperAnalyzer
from pysrc.papers.pubtrends_config import PubtrendsConfig
from pysrc.papers.db.loaders import Loaders

In [35]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

In [155]:
GROUPED_REFS_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/grouped_refs_validated/'
REFS_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/refs_validated/'
REFS_SELECTED_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/refs_selected/'
PUBTRENDS_EXPORT_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/pubtrends_export/'
CLUSTERING_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/clustering/'

LEVENSHTEIN_TITLE_DISTANCE_THRESHOLD = 5

In [153]:
PUBTRENDS_CONFIG = PubtrendsConfig(test=False)


def reload_exported_analyzer(path_to_archive, source='Pubmed'):
    """
    Load analysis data from json.gz archive generated by PubTrends.
    """
    with gzip.open(path_to_archive, 'rt', encoding='UTF-8') as zipfile:
        data = json.load(zipfile)

    loader, url_prefix = Loaders.get_loader_and_url_prefix(source, PUBTRENDS_CONFIG)
    analyzer = KeyPaperAnalyzer(loader, PUBTRENDS_CONFIG)
    analyzer.init(data)
    
    return analyzer

In [150]:
def reference_index(analyzer, review_pmid):
    reference_pmids = list(analyzer.citations_graph.successors(review_pmid))
    pubmed_data = analyzer.df[analyzer.df['id'].isin(reference_pmids)]
    pubmed_ref_search_index = {v.lower(): k for k, v in zip(pubmed_data['id'], pubmed_data['title'])}
    return pubmed_ref_search_index

In [159]:
def grobid2pmid(refs, reference_index, refs_selected=None):
    mapping = {}
    ref_mapped = {k: False for k in reference_index.keys()}

    for key, ref in refs.items():
        if ref.lower() in reference_index:
            pmid = reference_index[ref.lower()]
            mapping[key] = int(pmid)
            ref_mapped[ref.lower()] = True
            
#     print(mapping)
            
    refs_left = {key: ref for key, ref in refs.items() if key not in mapping}
#     print(refs_left)
#     print('---------')
#     print({k: v for k, v in ref_mapped.items() if not v})
    for pubmed_ref, mapped in ref_mapped.items():
        if not mapped:
            for grobid_id, ref in refs_left.items():
                if not ref:
                    continue
                distance = Levenshtein.distance(ref.lower(), pubmed_ref.lower())
                if distance < LEVENSHTEIN_TITLE_DISTANCE_THRESHOLD:
                    print(ref)
                    print(pubmed_ref)
                    match = input('Are these the same?')
                    if match == 'Y':
                        refs[grobid_id] = pubmed_ref
            if refs_selected:
                for grobid_if, ref in refs_selected.items():
                    distance = Levenshtein.distance(ref['title'].lower(), pubmed_ref.lower())
                    if distance < LEVENSHTEIN_TITLE_DISTANCE_THRESHOLD:
                        print(ref['title'])
                        print(pubmed_ref)
                        match = input('Are these the same?')
                        if match == 'Y':
                            refs[grobid_id] = pubmed_ref
#             print(pubmed_ref)
#             print(reference_index[pubmed_ref])
#             print()
            
#     input('Press any key to continue...')
            
    return mapping, refs

In [40]:
def export_to_json(data, filename):
    with open(filename, 'w') as f:
        json.dump(data, f, indent=4)

In [41]:
def pmid_clustering(grouped_refs, pmid_mapping, prefix=''):
    clustering = []
    for el in grouped_refs:
        if isinstance(el, dict):
            new_element = {
                'title': el['title'],
                'references': pmid_clustering(el['references'], pmid_mapping, prefix='> ')
            }
        else:
            new_element = None
            if el[0]:  # Some references do not have IDs due to parsing error
                grobid_id = el[0][1:]  # '#b0' -> 'b0'
                if grobid_id in pmid_mapping:
                    new_element = pmid_mapping[grobid_id]
    
        if new_element:
            clustering.append(new_element)
            
    return clustering

In [157]:
def obtain_clustering(file, save_clustering=True, save_references=False):
    pmid, ext = os.path.splitext(file)
    
    logging.info(f'{pmid}: loading file with grouped references')
    filename = os.path.join(GROUPED_REFS_FOLDER, file)
    with open(filename, 'r') as f:
        data = json.load(f)

    logging.info(f'{pmid}: loading file with PubTrends analyzer')
    analysis_file = os.path.join(PUBTRENDS_EXPORT_FOLDER, f'{pmid}.json.gz')
    analyzer = reload_exported_analyzer(analysis_file)
    
    logging.info(f'{pmid}: building reference index for matching titles and PMIDs')
    ref_index = reference_index(analyzer, pmid)
    
    logging.info(f'{pmid}: loading file with references')
    refs_file = os.path.join(REFS_FOLDER, file)
    with open(refs_file, 'r') as f:
        refs = json.load(f)
        
#     logging.info(f'{pmid}: loading file with selected references')
#     refs_selected_file = os.path.join(REFS_SELECTED_FOLDER, file)
#     with open(refs_selected_file, 'r') as f:
#         refs_selected = json.load(f)
        
    logging.info(f'{pmid}: building mapping between Grobid reference IDs and PMIDs')
    mapping, fixed_refs = grobid2pmid(refs, ref_index)
    n_pubmed_refs = analyzer.citations_graph.out_degree(pmid)
    full_mapping = len(mapping) == n_pubmed_refs
    print(f'{pmid}: {"[100%] " if full_mapping else ""}{len(mapping)} / {n_pubmed_refs} references mapped')
    
    logging.info(f'{pmid}: building clustering with PMIDs instead of Grobid IDs')
    clustering = pmid_clustering(data, mapping)
    clustering_file = os.path.join(CLUSTERING_FOLDER, file)
    
    if save_clustering:
        logging.info(f'{pmid}: exporting clustering')
        export_to_json(clustering, clustering_file)
        
    if save_references:
        logging.info(f'{pmid}: exporting references')
        export_to_json(fixed_refs, refs_file)

    logging.info(f'{pmid}: done\n')
    return len(mapping), n_pubmed_refs

In [160]:
if not os.path.exists(CLUSTERING_FOLDER):
    os.mkdir(CLUSTERING_FOLDER)

refs_mapped = 0
refs_total = 0
for file in sorted(os.listdir(GROUPED_REFS_FOLDER)):
    new_refs_mapped, new_refs_total = obtain_clustering(file)
    refs_mapped += new_refs_mapped
    refs_total += new_refs_total

2021-02-07 01:53:04,843 INFO: 26580716: loading file with grouped references
2021-02-07 01:53:04,845 INFO: 26580716: loading file with PubTrends analyzer
2021-02-07 01:53:05,568 INFO: 26580716: building reference index for matching titles and PMIDs
2021-02-07 01:53:05,572 INFO: 26580716: loading file with references
2021-02-07 01:53:05,574 INFO: 26580716: building mapping between Grobid reference IDs and PMIDs
2021-02-07 01:53:05,577 INFO: 26580716: building clustering with PMIDs instead of Grobid IDs
2021-02-07 01:53:05,579 INFO: 26580716: exporting clustering
2021-02-07 01:53:05,585 INFO: 26580716: done

2021-02-07 01:53:05,620 INFO: 26580717: loading file with grouped references
2021-02-07 01:53:05,623 INFO: 26580717: loading file with PubTrends analyzer


26580716: [100%] 152 / 152 references mapped


2021-02-07 01:53:06,673 INFO: 26580717: building reference index for matching titles and PMIDs
2021-02-07 01:53:06,676 INFO: 26580717: loading file with references
2021-02-07 01:53:06,678 INFO: 26580717: building mapping between Grobid reference IDs and PMIDs
2021-02-07 01:53:06,680 INFO: 26580717: building clustering with PMIDs instead of Grobid IDs
2021-02-07 01:53:06,682 INFO: 26580717: exporting clustering
2021-02-07 01:53:06,685 INFO: 26580717: done

2021-02-07 01:53:06,729 INFO: 26656254: loading file with grouped references
2021-02-07 01:53:06,735 INFO: 26656254: loading file with PubTrends analyzer


26580717: [100%] 91 / 91 references mapped


2021-02-07 01:53:07,437 INFO: 26656254: building reference index for matching titles and PMIDs
2021-02-07 01:53:07,440 INFO: 26656254: loading file with references
2021-02-07 01:53:07,442 INFO: 26656254: building mapping between Grobid reference IDs and PMIDs
2021-02-07 01:53:07,455 INFO: 26656254: building clustering with PMIDs instead of Grobid IDs
2021-02-07 01:53:07,456 INFO: 26656254: exporting clustering
2021-02-07 01:53:07,461 INFO: 26656254: done

2021-02-07 01:53:07,499 INFO: 26667849: loading file with grouped references
2021-02-07 01:53:07,501 INFO: 26667849: loading file with PubTrends analyzer


...

32020081: 174 / 178 references mapped


2021-02-07 01:53:42,465 INFO: 32042144: building reference index for matching titles and PMIDs
2021-02-07 01:53:42,469 INFO: 32042144: loading file with references
2021-02-07 01:53:42,472 INFO: 32042144: building mapping between Grobid reference IDs and PMIDs
2021-02-07 01:53:42,504 INFO: 32042144: building clustering with PMIDs instead of Grobid IDs
2021-02-07 01:53:42,506 INFO: 32042144: exporting clustering
2021-02-07 01:53:42,509 INFO: 32042144: done

2021-02-07 01:53:42,576 INFO: 32699292: loading file with grouped references
2021-02-07 01:53:42,578 INFO: 32699292: loading file with PubTrends analyzer


32042144: 146 / 204 references mapped


2021-02-07 01:53:43,305 INFO: 32699292: building reference index for matching titles and PMIDs
2021-02-07 01:53:43,309 INFO: 32699292: loading file with references
2021-02-07 01:53:43,312 INFO: 32699292: building mapping between Grobid reference IDs and PMIDs
2021-02-07 01:53:43,315 INFO: 32699292: building clustering with PMIDs instead of Grobid IDs
2021-02-07 01:53:43,320 INFO: 32699292: exporting clustering
2021-02-07 01:53:43,328 INFO: 32699292: done



32699292: 148 / 153 references mapped


In [161]:
refs_mapped / refs_total

0.9405851462865716

## Bulk PubTrends Export

In [ ]:
import gzip
import json
import logging
import os

from pysrc.papers.analyzer import KeyPaperAnalyzer
from pysrc.papers.pubtrends_config import PubtrendsConfig, DEFAULT_ANALYZER_SETTINGS
from pysrc.papers.db.loaders import Loaders
from pysrc.papers.db.search_error import SearchError
from pysrc.papers.plot.plotter import visualize_analysis
from pysrc.papers.progress import Progress
from pysrc.papers.utils import SORT_MOST_CITED, ZOOM_OUT, PAPER_ANALYSIS

In [ ]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

In [ ]:
TARGET_FOLDER = '/home/willenjoy/work/pubtrends/local/pubtrends_export/'

TARGET_PMIDS = [26667849, 26678314, 26688349, 26688350, 26580716, 26580717, 26656254, 26675821, 27834397, 27834398,
                27890914, 27916977, 27677859, 27677860, 27904142, 28003656, 29147025, 29170536, 28853444, 28920587,
                28792006, 28852220, 29213134, 29321682, 30578414, 30842595, 30644449, 30679807, 30108335, 30390028,
                30459365, 30467385, 31686003, 31806885, 31836872, 32005979, 31937935, 32020081, 32042144, 32699292]

SOURCE = 'Pubmed'
LIMIT = 500

In [ ]:
def export_analysis(pmid):
    logging.info(f'Started analysis for PMID {pmid}')
    
    ids = [pmid]
    query = f'Paper ID: {pmid}'
    
    # extracted from 'analyze_id_list' Celery task
    config = PubtrendsConfig(test=False)
    loader = Loaders.get_loader(SOURCE, config)
    analyzer = KeyPaperAnalyzer(loader, config)
    settings = DEFAULT_ANALYZER_SETTINGS
    try:
        # Fetch references at first
        ids = ids + analyzer.load_references(
            ids[0], limit=LIMIT
        )
        # And then expand
        ids = analyzer.expand_ids(
            ids, limit=LIMIT,
            expand_steps=settings.EXPAND_STEPS, expand_citations_sigma=settings.EXPAND_CITATIONS_SIGMA,
            expand_similarity_threshold=settings.EXPAND_SIMILARITY_THRESHOLD,
            current=1, task=None
        )

        analyzer.analyze_papers(ids, query, task=None)
    finally:
        loader.close_connection()

    dump = analyzer.dump()
    
    # export as JSON
    path = os.path.join(TARGET_FOLDER, f'{pmid}.json.gz')
    with gzip.open(path, 'w') as f:
        f.write(json.dumps(dump).encode('utf-8'))
    
    logging.info(f'Finished analysis for PMID {pmid}\n')

In [ ]:
TARGET_PMIDS = [49, 58, 59, 64]

In [ ]:
for pmid in TARGET_PMIDS:
    export_analysis(pmid)